# Linear Projection (GEMM): Math, Data Flow & CuTeDSL

## 1. Where Linear Projections Sit in Inference

During a single decode or prefill step, the transformer applies linear projections at two points per layer: the attention block (Q/K/V/O) and the MLP block (gate/up/down). Every one of these is a GEMM.

```
Input x ∈ ℝ^{B×T×4096}  (from RMSNorm)
  ①  Q = x @ W_Q.T         GEMM  [M=B·T, K=4096, N=4096]
  ②  K = x @ W_K.T         GEMM  [M=B·T, K=4096, N=1024]
  ③  V = x @ W_V.T         GEMM  [M=B·T, K=4096, N=1024]
  ④  ... attention ...
  ⑤  O = attn @ W_O.T      GEMM  [M=B·T, K=4096, N=4096]
  ⑥  gate = x₂ @ W_gate.T  GEMM  [M=B·T, K=4096, N=12288]
  ⑦  up   = x₂ @ W_up.T    GEMM  [M=B·T, K=4096, N=12288]
  ⑧  down = (gate⊙up) @ W_down.T  GEMM  [M=B·T, K=12288, N=4096]
Plus: lm_head = x_final @ W_head.T  [M=B·T, K=4096, N=151936]
```

GEMMs account for **~95% of all FLOPs** and **~90% of all weight bytes** in Qwen3-8B.

## 2. I/O Specification

**Table 1 — Q projection at decode (M=1) and prefill (M=2048):**

| Field | Decode | Prefill |
|---|---|---|
| Input x | [1, 1, 4096] BF16 | [B, T, 4096] BF16 |
| Weight W | [4096, 4096] BF16 (frozen) | same |
| Output Y | [1, 1, 4096] BF16 | [B, T, 4096] BF16 |
| FLOPs | 2·1·4096·4096 = 33.6M | 2·B·T·4096·4096 = 68.7G (B=1,T=2048) |
| Bytes read | 1·4096·2 + 4096²·2 = ~32 MB | T·4096·2 + 4096²·2 = ~32 MB |
| Intensity | ~0.5 FLOP/byte | ~340 FLOP/byte (T=2048) |
| Bound | Memory | Compute |

**Data lineage:**
- What produced the input: RMSNorm output (see RMSNorm notebook).
- What consumes the output: Q/K/V splitting → RoPE → attention.

## 3. The Formula

Standard linear layer (`torch.nn.Linear`) computes:

$$Y = X W^T \quad \text{where} \quad X \in \mathbb{R}^{M \times K},\ W \in \mathbb{R}^{N \times K},\ Y \in \mathbb{R}^{M \times N}$$

Element-wise:

$$Y_{m,n} = \sum_{k=0}^{K-1} X_{m,k} \cdot W_{n,k}$$

With bias (optional, Qwen3-8B projections have no bias):

$$Y_{m,n} = \sum_{k=0}^{K-1} X_{m,k} \cdot W_{n,k} + b_n$$

The computation is: M·N dot products, each of length K. Total FLOPs = 2·M·N·K (one multiply + one add per element of the inner product).

**Arithmetic intensity:**

$$I = \frac{2MNK}{(MK + NK + MN) \times 2\ \text{bytes}} \approx \frac{MNK}{(M+N)K} = \frac{MN}{M+N}$$

For fixed N=K (square weight, large M): I → M/2. For M=1 (decode): I → N/(N+1) ≈ 1 FLOP/byte → memory-bound.

## 4. Derivation: Why Y = X W^T

`torch.nn.Linear` stores weight as shape `[N, K]` (output_features, input_features). The forward pass computes `Y = X @ W.T` where each **row** of W is one "detector" — it computes the dot product of the input with that detector pattern.

- Row 0 of W asks: "how much does x look like pattern W[0,:]?"
- Row n of W asks: "how much does x look like pattern W[n,:]?"
- Y[m,n] = strength of input m's response to detector n

Equivalently: `Y = X @ W.T` is a set of M parallel dot products, each of length K, producing N outputs per input.

**Tiling rationale:**

If M threads compute Y[m,0] for m=0..M-1, each needs W[0,:] — all M threads fetch the same K weights from HBM. Tiling solves this:

> One block fetches one tile of W into SMEM, and all B_M threads reuse it: **B_M reuse factor per weight byte read**.

This is why larger BM tiles are better for memory efficiency: a BM=128 tile reads one W tile once and computes 128 output rows from it, amortizing the HBM→SMEM transfer cost 128×.

## 5. Symbol Table

| Symbol | Definition | Python / PyTorch | CuTeDSL | CUDA / Triton |
|---|---|---|---|---|
| X | Input activation matrix | `x: Tensor[M,K]` | `mX: cute.Tensor` | `const __nv_bfloat16* X` / `X_ptr` |
| W | Weight matrix (stored [N,K]) | `self.weight: Tensor[N,K]` | `mW: cute.Tensor` | `const __nv_bfloat16* W` / `W_ptr` |
| Y | Output matrix | `y: Tensor[M,N]` | `mOut: cute.Tensor` | `__nv_bfloat16* Y` / `Y_ptr` |
| M | Batch × sequence (rows) | `M = B * T` | tile dim BM | `int M` |
| N | Output dimension (cols of W) | `N = W.shape[0]` | tile dim BN | `int N` |
| K | Input dimension (inner) | `K = W.shape[1]` | tile dim BK | `int K` |
| BM,BN,BK | GEMM tile dimensions | not explicit | `BM=128, BN=128, BK=32` | `BLOCK_M`, `BLOCK_N`, `BLOCK_K` |
| rC | Register accumulator | `acc` (float32) | `rC = cute.partition_C(...)` | `float acc[...]` |
| sA,sB | SMEM tile buffers | N/A | `sA = cute.shared_memory(BM,BK)` | `__shared__ half sA[BM][BK]` |
| cp.async | Async HBM→SMEM copy | N/A | `cute.copy(gmem, smem)` (pipelined) | `__pipeline_memcpy_async` |
| ldmatrix | Load SMEM tile into regs | N/A | `cute.copy(smem_to_rmem_copy, sA, rA)` | `ldmatrix.sync.aligned.m8n8.x4` |
| MMA atom | One tensor core instruction | N/A | `SM89_16x8x16_F32BF16BF16F32_TN()` | `mma.sync.aligned.m16n8k16` |
| 4096 FLOPs/cycle | Output of one MMA | N/A | one `cute.gemm(tiled_mma,rA,rB,rC)` | one `mma.sync` |

In [ ]:
import torch
torch.manual_seed(0)

M, N, K = 4, 6, 8   # small example
X = torch.randn(M, K)
W = torch.randn(N, K)   # stored [N, K], represents N "detectors" each of length K

# ── Level 0: intent ────────────────────────────────────────────
# Y = linear_projection(X, W)

# ── Level 1: transpose convention ──────────────────────────────
# Y = X @ W.T

# ── Level 2: what each output element is ───────────────────────
# Y[m, n] = dot(X[m, :], W[n, :])

# ── Level 3: expand the dot product ────────────────────────────
# Y[m, n] = sum(X[m, k] * W[n, k] for k in range(K))

# ── Level 4: fully explicit triple loop ─────────────────────────
Y_loop = torch.zeros(M, N)
for m in range(M):
    for n in range(N):
        acc = 0.0
        for k in range(K):
            acc += X[m, k] * W[n, k]    # inner product element
        Y_loop[m, n] = acc

# ── Verify ──────────────────────────────────────────────────────
Y_torch = X @ W.T
print(f"X shape: {list(X.shape)}")
print(f"W shape: {list(W.shape)}  (N detectors, each of length K)")
print(f"Y shape: {list(Y_torch.shape)}")
print(f"Max diff (loop vs torch): {(Y_loop - Y_torch).abs().max():.2e}")
assert (Y_loop - Y_torch).abs().max() < 1e-5
print("✓ triple loop and torch.matmul agree")

print(f"\nFLOPs = 2 × M × N × K = 2 × {M} × {N} × {K} = {2*M*N*K}")
print(f"Bytes (read X+W, write Y) = (M·K + N·K + M·N) × 4 = {(M*K + N*K + M*N)*4}")
print(f"Arithmetic intensity = {2*M*N*K / ((M*K + N*K + M*N)*4):.2f} FLOP/byte")

## 6. Arithmetic Intensity Across All 7 GEMMs

RTX 4080 ridge = 151 FLOP/byte. Formula: I ≈ MN/(M+N) FLOP/byte (BF16, ignoring output bytes).

| Projection | N | K | Decode intensity (M=1) | Prefill intensity (M=2048) | Bottleneck |
|---|---|---|---|---|---|
| q_proj | 4096 | 4096 | ~0.5 F/B | ~340 F/B | decode:mem, prefill:compute |
| k_proj | 1024 | 4096 | ~0.5 F/B | ~168 F/B | decode:mem |
| v_proj | 1024 | 4096 | ~0.5 F/B | ~168 F/B | decode:mem |
| o_proj | 4096 | 4096 | ~0.5 F/B | ~340 F/B | decode:mem, prefill:compute |
| gate_proj | 12288 | 4096 | ~0.5 F/B | ~500 F/B | decode:mem, prefill:compute |
| up_proj | 12288 | 4096 | ~0.5 F/B | ~500 F/B | decode:mem, prefill:compute |
| down_proj | 4096 | 12288 | ~0.5 F/B | ~500 F/B | decode:mem, prefill:compute |

At decode, **ALL** projections are ~300× below the ridge. At prefill with T=2048, most are above.

**Weight bandwidth cost at decode (BF16):**

Weight sizes per layer: q_proj (32 MB) + k_proj (8 MB) + v_proj (8 MB) + o_proj (32 MB) + gate_proj (96 MB) + up_proj (96 MB) + down_proj (96 MB) = **368 MB/layer**

36 layers × 368 MB = **13.3 GB per token** at BF16. At 380 GB/s → ~35 ms/token minimum (memory wall).

This is why FP8/INT4 quantization matters for decode: halving weight bytes halves the memory wall directly.

## 7. Tiling Strategy for SM89

The tile sizes determine SMEM usage, register pressure, and occupancy.

**Formula for double-buffered SMEM:**

$$\text{SMEM} = 2 \times (B_M \times B_K + B_N \times B_K) \times 2\ \text{bytes}$$

**Standard config BM=128, BN=128, BK=32 (BF16):**
- Single buffer: (128×32 + 128×32)×2 = 16 KB
- Double buffer: 32 KB → 128 KB ÷ 32 KB = **4 CTAs per SM**

**Number of output tiles for q_proj prefill (M=2048, N=4096):**
- Grid: (M/BM, N/BN) = (16, 32) = **512 CTAs**
- 76 SMs × 4 CTAs = 304 active → ~2 waves → good utilization

**For decode M=1:**
- The BM=128 tile is 127/128 empty → GPU underutilized
- Split-K parallelizes over the K dimension instead: multiple CTAs each compute a partial sum over a K slice, then reduce
- This fills the SM grid at the cost of one extra reduction pass

**Occupancy levers:**
| Parameter | Effect |
|---|---|
| ↑ BM or BN | More reuse, more SMEM, fewer CTAs/SM |
| ↑ BK | More pipeline depth, more SMEM |
| ↑ pipeline stages | Hides HBM latency, multiplies SMEM |
| ↓ register count | More CTAs/SM (occupancy), less ILP |

In [ ]:
# Map each math operation to CuTeDSL primitives.
# This is pseudocode — run with cutedsl installed.

# Y[m,n] = sum_k X[m,k] * W[n,k]      ← the math
#
# In CuTeDSL, the kernel:
#   1. Partitions [M,N] output into [BM,BN] tiles  →  one CTA per tile
#   2. Loops over K in steps of BK                 →  k_tile loop
#   3. Each k_tile step:
#      a. Load X[row_tile, k_tile] → sA  (async, background)
#      b. Load W[col_tile, k_tile] → sB  (async, background)
#      c. sA → rA (ldmatrix)
#      d. sB → rB (ldmatrix)
#      e. rC += rA × rB              (MMA: mma.sync.aligned.m16n8k16)
#   4. Write rC → Y[row_tile, col_tile]

print("CuTeDSL GEMM primitive mapping:")
print()
print("  Math: sum_k X[m,k] * W[n,k]")
print("  ↓ partition M,N into BM=128, BN=128 tiles")
print("  ↓ each CTA owns one output tile [BM,BN]")
print()
print("  For each k-tile in range(K // BK):")
print("    cp.async  HBM→SMEM:  X[row_tile, k_tile] → sA  (non-blocking)")
print("    cp.async  HBM→SMEM:  W[col_tile, k_tile] → sB  (non-blocking)")
print("    pipeline_consumer_wait() — stall until tile lands")
print("    ldmatrix  SMEM→regs: sA → rA  (warp-level, tensor-core layout)")
print("    ldmatrix  SMEM→regs: sB → rB")
print("    mma.sync              rC += rA × rB  (4096 FLOPs, 1 instruction)")
print()
print("  Write: rC → Y[row_tile, col_tile]  (register → HBM, no extra barrier)")
print()
print("Key CuTeDSL calls:")
print("  cute.make_tiled_mma(SM89_16x8x16_F32BF16BF16F32_TN())")
print("  cute.make_pipeline(stages=2, ...)  ← double buffering")
print("  cute.copy(gmem_a, smem_a)          ← cp.async under the hood")
print("  cute.copy(smem_to_rmem_copy, sA, rA) ← ldmatrix under the hood")
print("  cute.gemm(tiled_mma, rA, rB, rC)   ← mma.sync under the hood")

## 8. What Q, K, V Are — The Projection Output

After the GEMM `Y = X @ W.T`, reshape and split heads:

- After q_proj: Y ∈ ℝ^{B×T×4096} → reshape → **Q ∈ ℝ^{B×T×32×128}** (32 heads, head_dim=128)
- After k_proj: **K ∈ ℝ^{B×T×8×128}** (8 KV-heads; GQA: each K head serves 4 Q heads)
- After v_proj: **V ∈ ℝ^{B×T×8×128}**

The GEMM output is a flat `[B·T, head_dim×num_heads]` tensor. The attention kernel reshapes it to per-head tensors.

**Qwen3-specific: QK-norm** is applied after reshape — per-head RMSNorm on Q and K before computing attention scores. This stabilizes training with GQA.

```
x                  [B, T, 4096]
  → q_proj GEMM    [B·T, 4096]
  → reshape         [B, T, 32, 128]   # 32 Q heads
  → QK-norm         per-head RMSNorm
  → RoPE            rotary position embedding
  → attention QK^T  [B, 32, T, T_cache]
```

## 9. Production GEMM Kernels

| Library | API | What it does |
|---|---|---|
| cuBLASLt | `cublasLtMatmul` | Default backend for torch.nn.Linear; black box tile selection |
| CUTLASS 3.x | `GemmUniversal` | Exposes full tiling hierarchy; used in this project |
| FlashInfer | `gemm.bmm_fp8` | Batched FP8 GEMM with per-tensor/block scales |
| vLLM | Marlin kernel (`csrc/ops/gemm_kernels.cu`) | W4A16 quantized GEMM; W8A8 FP8 via cuBLASLt |
| SGLang | Calls vLLM/FlashInfer kernels | No custom GEMM; relies on upstream |
| TRT-LLM | FusedMHA plugin | QKV projection + attention + O projection fused into one TRT op |

**Key fusion: QKV fused projection**

Instead of 3 separate GEMMs (Q, K, V), compute:
```
[Q; K; V] = X @ [W_Q; W_K; W_V].T   shape [M, 4096+1024+1024] = [M, 6144]
```
One GEMM instead of 3, 3× better SM utilization at decode. Used in vLLM's `QKVParallelLinear`.

**Kernel fusion: GEMM + RMSNorm epilogue**

After computing `Y = X @ W.T` in registers (rC), apply RMSNorm before writing to HBM:
- Compute row RMS in registers
- Normalize rC in registers
- Write normalized output once

Saves one full HBM write+read of the output tensor. Particularly valuable at decode where memory bandwidth is the bottleneck.

## One-Sentence Takeaway

Linear projection is the dominant cost in Qwen3-8B (95% of FLOPs, 13.7 GB of weight reads per token) with a 680× gap in arithmetic intensity between decode (0.5 FLOP/byte, memory-bound, FP8 quantization is the only real lever) and prefill at T=2048 (340 FLOP/byte, compute-bound, tensor-core utilization via tiling and double buffering is what matters).